# NullFusion — Null-Space Conditional Fusion (proposal7)

**Goal:** beat the CAVE x4 / Nikon D700 SRF SOTA (FeINFN 52.47 / BDT 52.30 dB) with a *novel*, Q1-grade architecture.

`X_hat = pinv(yH, yM) + P_N( f_theta(conditioning) )`

The observation-consistent component is **solved in closed form** (exact `A(X_hat)==[yH;yM]` is an algebraic identity, ~1e-5), and the network only ever fills the **null space** the sensors provably cannot see — so it *cannot hallucinate* the observable part (P1 admissible ambiguity).

In [ ]:
import subprocess, glob, os, shutil

# Recursively find .whl files under /kaggle/input/
whls = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.whl'):
            whls.append(os.path.join(root, f))
whls.sort()
print(f'Found wheels: {whls}')
if not whls:
    raise RuntimeError('No torch wheels found')
for w in whls:
    base = os.path.basename(w)
    fixed = base.replace('cu121-cp312', '+cu121-cp312')
    dst = os.path.join('/tmp', fixed)
    shutil.copy2(w, dst)
    print(f'Installing {dst}')
    subprocess.check_call(['pip', 'install', '--force-reinstall', '--no-deps', dst])
print('torch install done')

In [ ]:
import torch, sys
print('torch', torch.__version__)
assert torch.cuda.is_available(), 'CUDA not available'
p = torch.cuda.get_device_properties(0)
print(f'gpu {p.name} {p.total_memory/2**30:.1f}GB sm_{p.major}{p.minor}')

In [ ]:
!pip install -q scipy scikit-image matplotlib

In [ ]:
import os, sys, glob

# Find the repo root by looking for proposal7/nullfusion/model.py
REPO = None
for root, dirs, files in os.walk('/kaggle/input'):
    if os.path.exists(os.path.join(root, 'proposal7', 'nullfusion', 'model.py')):
        REPO = root
        break
    if os.path.exists(os.path.join(root, 'common', 'hsifusion', '__init__.py')):
        REPO = root
        break

# If repo found as zip, extract it
if REPO is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip'):
                import zipfile
                REPO = '/kaggle/working/repo'
                os.makedirs(REPO, exist_ok=True)
                with zipfile.ZipFile(os.path.join(root, f)) as zf:
                    zf.extractall(REPO)
                break
        if REPO:
            break

if REPO is None:
    raise RuntimeError('repo not found under /kaggle/input/')

sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'common'))
print('REPO =', REPO)

from proposal7.nullfusion import NullFusionNet, NullFusionConfig
print('NullFusionNet import OK')

## 1. CAVE dataset

In [ ]:
from hsifusion.io_utils import discover_dataset, available_splits, find_pairs, list_hsi

spec_root = discover_dataset(["CAVE"], required=True)
splits = available_splits(spec_root)
print("CAVE splits:", splits)
train_pairs = find_pairs(spec_root, splits.get("Train", "Train"))
test_scenes = list_hsi(spec_root, splits.get("Test", "Test"))
print(f"train pairs = {len(train_pairs)}   test scenes = {len(test_scenes)}")

## 2. Train NullFusionNet (CAVE x4, Nikon D700 SRF)

In [ ]:
from proposal7.nullfusion.train_sota import TrainConfig, train

cfg = TrainConfig()     # v2 defaults: width=128, depth=12, cross-attn, refine

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
train(cfg, device)

## 3. Results

In [ ]:
import json, os
if os.path.exists('sota_results.json'):
    with open('sota_results.json') as f:
        res = json.load(f)
    print('protocol:', res['protocol'])
    print('mean    :', {k: round(v, 3) for k, v in res['mean'].items()})
    print('params  :', res['nparams'])